# Aircraft Vision Queue Worker — Colab

Phase 11 Sprint 11.3 — long-running Colab worker that polls
`vision_index_jobs` for `status='queued'` rows and processes them.
Replaces the one-off backfill notebook for ongoing operations.

**Required Colab Secrets** (sidebar key icon, notebook access toggled ON):
- `HF_TOKEN` — HuggingFace token (read scope)
- `SUPABASE_URL` — Supabase project URL
- `SUPABASE_SERVICE_ROLE_KEY` — service-role key

**Runtime:** L4 (24 GB VRAM) High-RAM works perfectly. T4 also works if you bump GPU_BATCH=1.

**Heartbeat:** the worker upserts `vision_worker_heartbeat` every 30s. If you halt cell 5 cleanly, status flips to `'stopping'` and the Modal fallback sweep cron will pick up any stuck jobs within 5 min.

**Cost:** $10/mo Colab Pro flat. No per-job cost.

In [ ]:
# === Cell 1 — setup ===
# Run once per Colab session.
# numpy>=2.0 is pinned because transformers (pulled in by colpali-engine) ships
# wheels compiled against the NumPy 2.x C ABI; without this Cell 3 errors with
# "numpy.dtype size changed, may indicate binary incompatibility" when loading
# ColQwen2 (Phase 12 activation, 2026-05-09).
!pip install -q torch==2.4.1 torchvision==0.19.1 colpali-engine==0.3.5 'accelerate>=1.0,<2.0' 'numpy>=2.0,<3.0' pdf2image pillow supabase==2.30.0 requests
!apt-get -qq install -y poppler-utils > /dev/null
import torch
import uuid
WORKER_ID = 'colab-' + str(uuid.uuid4())[:8]
POLL_INTERVAL_SECONDS = 15
HEARTBEAT_INTERVAL_SECONDS = 30
MAX_BATCH_SIZE = 8
GPU_BATCH = 4
DPI = 180
print(f"WORKER_ID = {WORKER_ID}")
print(f"CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# === Cell 2 — secrets + Supabase ===
import os
from google.colab import userdata
from supabase import create_client

HF_TOKEN = userdata.get('HF_TOKEN')
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_SERVICE_ROLE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

# Sanity probe
queued = sb.table('vision_index_jobs').select('id', count='exact').eq('status', 'queued').limit(1).execute()
print(f"Connected. Currently queued jobs: {queued.count}")

In [ ]:
# === Cell 3 — load ColQwen2 ===
import time
from colpali_engine.models import ColQwen2, ColQwen2Processor

MODEL_NAME = 'vidore/colqwen2-v1.0'
MODEL_LABEL = 'colqwen2'
SUMMARY_DIM = 128
MAX_PATCHES_PER_PAGE = 64

_t = time.time()
model = ColQwen2.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda', token=HF_TOKEN).eval()
processor = ColQwen2Processor.from_pretrained(MODEL_NAME, token=HF_TOKEN)
print(f"ColQwen2 loaded in {time.time()-_t:.1f}s")

In [ ]:
# === Cell 4 — helpers (queue worker version) ===
import io
import traceback
import requests
from PIL import Image
from pdf2image import convert_from_bytes

# ─── ColQwen2 inference (same as backfill notebook) ──────────────────

def embed_pil_images(images):
    results = []
    for start in range(0, len(images), GPU_BATCH):
        batch = images[start:start + GPU_BATCH]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        with torch.no_grad():
            processed = processor.process_images(batch).to(model.device)
            page_embeds = model(**processed)
        for i in range(page_embeds.shape[0]):
            emb = page_embeds[i]
            summary = emb.mean(dim=0).to(torch.float32).cpu().tolist()
            patches_t = emb[:MAX_PATCHES_PER_PAGE].to(torch.float32).cpu()
            patches = [row.tolist() for row in patches_t]
            assert len(summary) == SUMMARY_DIM
            for r in patches: assert len(r) == SUMMARY_DIM
            results.append({'summary_vector': summary, 'patch_vectors': patches, 'patch_count': len(patches)})
    return results

# ─── Job claim/complete/fail ──────────────────────────────────────────

def claim_job():
    """Atomically claim the oldest queued job. Returns claimed row or None.
    Race-safe: if two workers race, only one wins via the WHERE status='queued' guard."""
    # supabase-py doesn't expose UPDATE...FROM(SELECT) directly; we do it in two steps
    # with optimistic concurrency: read oldest queued, attempt to flip to running.
    candidate = sb.table('vision_index_jobs').select('*').eq('status', 'queued').order('created_at').limit(1).execute()
    if not candidate.data:
        return None
    job = candidate.data[0]
    # Try to flip status='queued' → 'running' atomically.
    flipped = sb.table('vision_index_jobs').update({
        'status': 'running',
        'gpu_host': 'colab',
        'started_at': 'now()',
    }).eq('id', job['id']).eq('status', 'queued').execute()
    if not flipped.data:
        # Lost the race to another worker; try again next tick.
        return None
    return flipped.data[0]

def complete_job(job_id, success_count, fail_count):
    sb.table('vision_index_jobs').update({
        'status': 'completed',
        'completed_at': 'now()',
        'metadata': {'pages_indexed': success_count, 'pages_failed': fail_count, 'worker_id': WORKER_ID},
    }).eq('id', job_id).execute()

def fail_job(job_id, error_message):
    sb.table('vision_index_jobs').update({
        'status': 'failed',
        'completed_at': 'now()',
        'error_message': error_message[:4000],
        'metadata': {'worker_id': WORKER_ID},
    }).eq('id', job_id).execute()

# ─── Heartbeat ─────────────────────────────────────────────────────────

def heartbeat(status='idle', jobs_processed=0, last_job_id=None, last_error=None):
    row = {
        'worker_id': WORKER_ID,
        'gpu_host': 'colab',
        'status': status,
        'last_seen_at': 'now()',
        'jobs_processed_total': jobs_processed,
    }
    if last_job_id is not None:
        row['last_job_id'] = last_job_id
    if last_error is not None:
        row['last_error'] = last_error[:4000]
    sb.table('vision_worker_heartbeat').upsert(row, on_conflict='worker_id').execute()

# ─── Process job pages ────────────────────────────────────────────────

def process_job_pages(job):
    """Embed every page referenced by the job. Returns (success_count, fail_count)."""
    org_id = job['organization_id']
    page_ids = job['vision_page_ids']
    # Fetch the page rows
    pages_resp = sb.table('vision_pages').select('*').in_('id', page_ids).is_('deleted_at', 'null').execute()
    pages = [p for p in (pages_resp.data or []) if p['status'] in ('pending', 'rendering', 'embedding')]
    if not pages:
        return (0, 0)

    # Mark each page as 'embedding' (idempotent)
    for p in pages:
        sb.table('vision_pages').update({'status': 'embedding'}).eq('id', p['id']).execute()

    # Group pages by source_document_id so we can render the parent PDF once.
    by_doc = {}
    for p in pages:
        by_doc.setdefault(p['source_document_id'], []).append(p)

    success, fail = 0, 0
    for doc_id, doc_pages in by_doc.items():
        doc_pages.sort(key=lambda p: p['page_number'])
        # Pull doc.file_path so we can download the PDF
        doc_resp = sb.table('documents').select('file_path').eq('id', doc_id).maybe_single().execute()
        if not doc_resp or not doc_resp.data or not doc_resp.data.get('file_path'):
            for p in doc_pages:
                sb.table('vision_pages').update({'status': 'failed', 'error_message': 'doc not found / no file_path'}).eq('id', p['id']).execute()
                fail += 1
            continue
        try:
            pdf_bytes = sb.storage.from_('documents').download(doc_resp.data['file_path'])
            pil_pages = convert_from_bytes(pdf_bytes, dpi=DPI, fmt='png')
        except Exception as e:
            for p in doc_pages:
                sb.table('vision_pages').update({'status': 'failed', 'error_message': f'pdf: {e}'}).eq('id', p['id']).execute()
                fail += 1
            continue

        # Embed only the page numbers this job asked for.
        wanted_imgs = []
        wanted_pages = []
        for p in doc_pages:
            if p['page_number'] < len(pil_pages):
                wanted_imgs.append(pil_pages[p['page_number']])
                wanted_pages.append(p)
            else:
                sb.table('vision_pages').update({'status': 'failed', 'error_message': f'page_number {p["page_number"]} out of range ({len(pil_pages)} pages in PDF)'}).eq('id', p['id']).execute()
                fail += 1

        if not wanted_imgs:
            continue

        try:
            embeds = embed_pil_images(wanted_imgs)
        except Exception as e:
            for p in wanted_pages:
                sb.table('vision_pages').update({'status': 'failed', 'error_message': f'gpu: {e}'}).eq('id', p['id']).execute()
                fail += len(wanted_pages)
            continue

        for n, p in enumerate(wanted_pages):
            er = embeds[n]
            try:
                sb.table('vision_embeddings').insert({
                    'organization_id': org_id,
                    'vision_page_id': p['id'],
                    'model_used': MODEL_LABEL,
                    'embedding_dim': SUMMARY_DIM,
                    'summary_vector': er['summary_vector'],
                    'patch_vectors': {'patches': er['patch_vectors']},
                    'patch_count': er['patch_count'],
                }).execute()
                sb.table('vision_pages').update({
                    'status': 'indexed', 'vision_index_id': p['id'],
                    'vision_model': MODEL_LABEL, 'embedded_at': 'now()',
                }).eq('id', p['id']).execute()
                success += 1
            except Exception as e:
                sb.table('vision_pages').update({'status': 'failed', 'error_message': str(e)[:1000]}).eq('id', p['id']).execute()
                fail += 1

    return (success, fail)

print('Helpers loaded.')

In [ ]:
# === Cell 5 — main poll loop ===
# Stop this cell to halt cleanly. Heartbeat thread will mark status='stopping' on next tick.
import threading
import time

stop_flag = {'stop': False}
jobs_done = 0
current_job_id = None
last_error = None

def heartbeat_thread():
    while not stop_flag['stop']:
        try:
            heartbeat(
                status='busy' if current_job_id else 'idle',
                jobs_processed=jobs_done,
                last_job_id=current_job_id,
                last_error=last_error,
            )
        except Exception as e:
            print(f'  [hb-err] {e}')
        time.sleep(HEARTBEAT_INTERVAL_SECONDS)

t = threading.Thread(target=heartbeat_thread, daemon=True)
t.start()

print(f'Worker {WORKER_ID} starting poll loop (poll={POLL_INTERVAL_SECONDS}s, hb={HEARTBEAT_INTERVAL_SECONDS}s)...')
print('Stop the cell to halt cleanly.')

try:
    while True:
        job = claim_job()
        if job is None:
            current_job_id = None
            time.sleep(POLL_INTERVAL_SECONDS)
            continue

        current_job_id = job['id']
        n_pages = len(job.get('vision_page_ids') or [])
        print(f'\n[{jobs_done+1}] Processing job {current_job_id[:8]}... pages={n_pages}')

        try:
            t0 = time.time()
            success, fail = process_job_pages(job)
            elapsed = time.time() - t0
            complete_job(current_job_id, success, fail)
            jobs_done += 1
            last_error = None
            print(f'  [done] success={success} fail={fail} elapsed={elapsed:.1f}s')
        except Exception as e:
            tb = traceback.format_exc(limit=10)
            last_error = str(e)
            fail_job(current_job_id, f'{e}\n{tb[:2000]}')
            print(f'  [fail] {e}')
finally:
    stop_flag['stop'] = True
    try:
        heartbeat(status='stopping', jobs_processed=jobs_done, last_job_id=current_job_id, last_error=last_error)
    except Exception:
        pass
    print(f'\nWorker {WORKER_ID} stopped. Total jobs processed: {jobs_done}')

In [ ]:
# === Cell 6 — verify (run after halting cell 5) ===
# Worker heartbeat row
hb = sb.table('vision_worker_heartbeat').select('*').eq('worker_id', WORKER_ID).maybe_single().execute()
if hb and hb.data:
    print('Heartbeat row:')
    print(f"  worker_id: {hb.data['worker_id']}")
    print(f"  status: {hb.data['status']}")
    print(f"  last_seen_at: {hb.data['last_seen_at']}")
    print(f"  jobs_processed_total: {hb.data['jobs_processed_total']}")
else:
    print('No heartbeat row found.')

# Job status counts
for st in ('queued', 'running', 'completed', 'failed'):
    r = sb.table('vision_index_jobs').select('id', count='exact').eq('status', st).limit(1).execute()
    print(f"  jobs status={st}: {r.count}")